# 👤 EBLET Individual v2.0: Tu Perfil de Bienestar Laboral

## 🎯 Objetivo
Clasificar a cada persona en uno de los 6 perfiles de bienestar laboral
a partir de las **23 preguntas** de EBLET-Lite v2.0.

## 📋 Los 6 Perfiles

| Perfil | Descripción | Cultura Típica |
|--------|-------------|----------------|
| 🟢 Flourishing | Rendimiento óptimo y bienestar pleno | Clan, Adhocracia |
| 🟡 Estable Neutro | Zona de confort, mejorable | Cualquiera |
| 🟠 Quemado Activo | Agotamiento por exceso de carga | Mercado |
| 🔵 Aburrido Crónico | Falta de estimulación y retos | Jerárquica |
| 🔴 Crítico Dual | Agotado Y aburrido - situación grave | Mercado + Jerárquica |
| ⚫ Vuelo Inminente | Buscando activamente otra oportunidad | Cualquiera |



In [1]:

# IMPORTS Y CONFIGURACIÓN


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

import os
import sys

RAIZ_PROYECTO = r"C:\Users\torre\OneDrive\Escritorio\EBLET-People-Analytics\Python"
sys.path.insert(0, os.path.join(RAIZ_PROYECTO, "src"))

from encuesta_lite import (
    PREGUNTAS_LITE, 
    TEXTO_PREGUNTAS, 
    calcular_kpis_lite,
    validar_respuestas,
    generar_texto_formulario,
    PREGUNTAS_ORDENADAS
)
from clasificador_individual import (
    PERFILES, 
    BENCHMARK_PERCENTILES,
    clasificar_individuo, 
    clasificar_dataframe,
    generar_informe_individual
)

print("✅ Notebook 6 v2.0 cargado correctamente")
print(f"📋 EBLET-Lite v2.0: {len(PREGUNTAS_ORDENADAS)} preguntas")
print(f"👤 Perfiles disponibles: {len(PERFILES)}")
print(f"🏛️  Culturas CVF: Adhocracia, Clan, Mercado, Jerárquica (2 preguntas c/u)")
print(f"📊 Benchmark: 12,500 empleados para cálculo de percentiles")

✅ Notebook 6 v2.0 cargado correctamente
📋 EBLET-Lite v2.0: 23 preguntas
👤 Perfiles disponibles: 6
🏛️  Culturas CVF: Adhocracia, Clan, Mercado, Jerárquica (2 preguntas c/u)
📊 Benchmark: 12,500 empleados para cálculo de percentiles


## 1. 📝 Las 23 Preguntas de EBLET-Lite 

La encuesta se divide en 5 dimensiones + cultura percibida:

In [2]:

# MOSTRAR LAS 23 PREGUNTAS AGRUPADAS


dimension_info = {
    "burnout": ("🔥 Burnout", "#e74c3c"),
    "boreout": ("😴 Boreout", "#3498db"),
    "bienestar": ("💚 Bienestar", "#2ecc71"),
    "rotacion": ("🔄 Rotación", "#e67e22"),
    "cultura_cvf": ("🏛️ Cultura CVF", "#f39c12")
}

#print("="*80)
print("📝 EBLET-Lite v2.0: 23 PREGUNTAS")
#print("="*80)
print("\nEscala Likert: 1 (Totalmente en desacuerdo) → 5 (Totalmente de acuerdo)\n")

secciones = {
    "🔥 BURNOUT": [16, 19, 23, 28],
    "😴 BOREOUT": [37, 39, 41, 43],
    "💚 BIENESTAR": [45, 46, 47, 48],
    "🔄 ROTACIÓN": [57, 58, 59],
    "🏛️ CULTURA - Innovación (Adhocracia)": [65, 66],
    "🏛️ CULTURA - Colaboración (Clan)": [67, 68],
    "🏛️ CULTURA - Resultados (Mercado)": [69, 70],
    "🏛️ CULTURA - Normas (Jerarquía)": [71, 72]
}

num = 1
for seccion, preguntas in secciones.items():
    print(f"\n--- {seccion} ---")
    for q in preguntas:
        print(f"{num:2d}. {TEXTO_PREGUNTAS[q]}")
        num += 1

#print("\n" + "="*80)
print("💡 Duración estimada: 8 minutos")
print("🎯 Output: Perfil predominante + Cultura percibida + Percentiles")
#print("="*80)

📝 EBLET-Lite v2.0: 23 PREGUNTAS

Escala Likert: 1 (Totalmente en desacuerdo) → 5 (Totalmente de acuerdo)


--- 🔥 BURNOUT ---
 1. Me siento emocionalmente agotado/a por mi trabajo.
 2. Trabajar todo el dia es un verdadero esfuerzo.
 3. He desarrollado una actitud distante hacia mi trabajo.
 4. Me he vuelto menos entusiasta con mi trabajo.

--- 😴 BOREOUT ---
 5. Mi trabajo me resulta monotono y repetitivo.
 6. Tengo la sensacion de que mi trabajo carece de sentido.
 7. A menudo me aburro en el trabajo.
 8. Siento que estoy infrautilizado/a en mis funciones.

--- 💚 BIENESTAR ---
 9. Me he sentido alegre y de buen humor.
10. Me he sentido tranquilo/a y relajado/a.
11. Me he sentido activo/a y vigoroso/a.
12. He tenido energia para hacer las cosas del dia a dia.

--- 🔄 ROTACIÓN ---
13. Estoy buscando activamente otro empleo.
14. Es probable que cambie de empresa el proximo ano.
15. A veces pienso en dejar mi trabajo.

--- 🏛️ CULTURA - Innovación (Adhocracia) ---
16. En mi organizacion se fo

## 2. 🧪 Casos de Prueba

Vamos a clasificar 5 personas con perfiles diferentes en culturas distintas.
Cada caso incluye las **23 preguntas** completas.

In [3]:

# 5 CASOS DE PRUEBA (con 23 preguntas)


casos_prueba = {
    "Ana (Flourishing @ Clan)": {
        # Burnout (4)
        'q16': 2, 'q19': 1, 'q23': 2, 'q28': 2,
        # Boreout (4)
        'q37': 2, 'q39': 1, 'q41': 2, 'q43': 1,
        # Bienestar (4)
        'q45': 5, 'q46': 4, 'q47': 5, 'q48': 5,
        # Rotación (3)
        'q57': 1, 'q58': 1, 'q59': 1,
        # Cultura (8)
        'q65': 3, 'q66': 3, 'q67': 5, 'q68': 5,
        'q69': 2, 'q70': 2, 'q71': 2, 'q72': 2
    },
    "Carlos (Estable @ Adhocracia)": {
        'q16': 3, 'q19': 3, 'q23': 2, 'q28': 3,
        'q37': 3, 'q39': 2, 'q41': 3, 'q43': 2,
        'q45': 3, 'q46': 3, 'q47': 3, 'q48': 4,
        'q57': 2, 'q58': 3, 'q59': 2,
        'q65': 5, 'q66': 5, 'q67': 3, 'q68': 3,
        'q69': 3, 'q70': 3, 'q71': 2, 'q72': 2
    },
    "María (Quemada @ Mercado)": {
        'q16': 5, 'q19': 5, 'q23': 4, 'q28': 4,
        'q37': 2, 'q39': 2, 'q41': 2, 'q43': 2,
        'q45': 2, 'q46': 1, 'q47': 2, 'q48': 2,
        'q57': 3, 'q58': 3, 'q59': 3,
        'q65': 2, 'q66': 2, 'q67': 2, 'q68': 2,
        'q69': 5, 'q70': 5, 'q71': 3, 'q72': 3
    },
    "David (Aburrido @ Jerárquica)": {
        'q16': 2, 'q19': 2, 'q23': 2, 'q28': 2,
        'q37': 5, 'q39': 4, 'q41': 5, 'q43': 5,
        'q45': 2, 'q46': 2, 'q47': 2, 'q48': 2,
        'q57': 4, 'q58': 4, 'q59': 4,
        'q65': 2, 'q66': 2, 'q67': 2, 'q68': 2,
        'q69': 3, 'q70': 3, 'q71': 5, 'q72': 5
    },
    "Laura (Crítica @ Mercado)": {
        'q16': 5, 'q19': 5, 'q23': 4, 'q28': 5,
        'q37': 5, 'q39': 4, 'q41': 5, 'q43': 4,
        'q45': 1, 'q46': 1, 'q47': 1, 'q48': 1,
        'q57': 5, 'q58': 5, 'q59': 5,
        'q65': 2, 'q66': 2, 'q67': 1, 'q68': 1,
        'q69': 5, 'q70': 5, 'q71': 3, 'q72': 3
    }
}

# Calcular KPIs y clasificar cada caso
resultados = []
for nombre, respuestas in casos_prueba.items():
    valido, msg = validar_respuestas(respuestas)
    if not valido:
        print(f"⚠️ {nombre}: {msg}")
        continue
    
    kpis = calcular_kpis_lite(respuestas)
    perfil = clasificar_individuo(kpis)
    
    resultados.append({
        "Nombre": nombre,
        "Perfil": perfil["nombre"],
        "Cultura": kpis.get("cultura_dominante", "-"),
        "Burnout": round(kpis["burnout"], 2),
        "Boreout": round(kpis["boreout"], 2),
        "Bienestar": round(kpis["bienestar"], 2),
        "Rotación": round(kpis["rotacion"], 2),
        "_kpis": kpis,
        "_perfil": perfil
    })

df_resultados = pd.DataFrame(resultados)

print("📊 CLASIFICACIÓN DE LOS 5 CASOS DE PRUEBA")

print(df_resultados[["Nombre", "Perfil", "Cultura", "Burnout", "Boreout", "Bienestar", "Rotación"]].to_string(index=False))

📊 CLASIFICACIÓN DE LOS 5 CASOS DE PRUEBA
                       Nombre             Perfil    Cultura  Burnout  Boreout  Bienestar  Rotación
     Ana (Flourishing @ Clan)      🟢 Flourishing       Clan     1.75     1.50       4.75      1.00
Carlos (Estable @ Adhocracia)   🟡 Estable Neutro Adhocracia     2.75     2.50       3.25      2.33
    María (Quemada @ Mercado)   🟠 Quemado Activo    Mercado     4.50     2.00       1.75      3.00
David (Aburrido @ Jerárquica) 🔵 Aburrido Crónico Jerarquica     2.00     4.75       2.00      4.00
    Laura (Crítica @ Mercado)     🔴 Crítico Dual    Mercado     4.75     4.50       1.00      5.00


## 3. 📋 Informe Individual Detallado

 **David (Aburrido en empresa Jerárquica)**.


In [4]:

# INFORME DETALLADO DE UN CASO


caso_elegido = "David (Aburrido @ Jerárquica)"
caso = next(c for c in resultados if c["Nombre"] == caso_elegido)
respuestas_raw = casos_prueba[caso_elegido]
print(generar_informe_individual(caso["_kpis"], caso["_perfil"], respuestas_raw))



  🎯 TU PERFIL DE BIENESTAR LABORAL


                    🔵 Aburrido Crónico

  💬 No estás cansado, pero tu trabajo no te estimula lo suficiente. Sientes que tus capacidades no se aprovechan plenamente.


  📊 TUS INDICADORES PRINCIPALES

║    🔥 Burnout   :  40% ████████░░░░░░░░░░░░ 🟢
║    😴 Boreout   :  95% ███████████████████░ 🔴
║    💚 Bienestar :  40% ████████░░░░░░░░░░░░ 🔴
║

  🏛️ CULTURA DE TU ORGANIZACIÓN (según tu percepción)

    🔵 Adhocracia  : 2/5 ████████░░░░░░░░░░░░   
    🟢 Clan        : 2/5 ████████░░░░░░░░░░░░   
    🟠 Mercado     : 3/5 ████████████░░░░░░░░   
    🟡 Jerarquica  : 5/5 ████████████████████ ⭐

    🎯 Cultura predominante: JERARQUICA
       (Procesos, estabilidad, normas)


  🔗 INDICADORES COMPLEMENTARIOS

    🔀 Intención de cambio laboral: Elevada 🔴
                   ████████████████░░░░

    💼 Compromiso con la organización: Bajo 🔴

    📈 Tu posición respecto al benchmark (12,500 empleados):
       🔥 Burnout   : Percentil  23 (BAJO)
       😴 Boreout   : Pe

## 4. 🕸️ Visualización Radar de los 5 Casos

Comparamos los perfiles en un gráfico de radar (con las 4 dimensiones):

In [5]:

# RADAR CHART: 5 CASOS


fig = go.Figure()

color_map = {
    "🟢 Flourishing": "#2ecc71",
    "🟡 Estable Neutro": "#f1c40f",
    "🟠 Quemado Activo": "#e67e22",
    "🔵 Aburrido Crónico": "#3498db",
    "🔴 Crítico Dual": "#e74c3c",
    "⚫ Vuelo Inminente": "#34495e"
}

labels = ["Bienestar", "Sin Burnout", "Sin Boreout", "Retención"]

for _, row in df_resultados.iterrows():
    valores = [
        row["Bienestar"],
        5 - row["Burnout"],
        5 - row["Boreout"],
        5 - row["Rotación"]
    ]
    valores_cerrados = valores + [valores[0]]
    labels_cerrados = labels + [labels[0]]
    
    color = color_map.get(row["Perfil"], "#95a5a6")
    
    fig.add_trace(go.Scatterpolar(
        r=valores_cerrados,
        theta=labels_cerrados,
        fill='toself',
        name=row["Nombre"],
        line_color=color,
        opacity=0.6,
        line=dict(width=2)
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 5]), bgcolor="white"),
    title="🕸️ Perfil Multidimensional de Bienestar Laboral",
    showlegend=True,
    height=650,
    width=850,
    legend=dict(orientation="h", yanchor="bottom", y=-0.15, xanchor="center", x=0.5)
)
fig.show()

print("""
💡 INTERPRETACIÓN:
- Cuanto más grande el área, más saludable el perfil
- Ana (verde) tiene el área máxima - Flourishing
- Laura (rojo) tiene el área mínima - Crítico
- David (azul) destaca en "Sin Burnout" pero es bajo en "Sin Boreout"
- María (naranja) destaca en "Sin Boreout" pero es bajo en "Sin Burnout"
""")


💡 INTERPRETACIÓN:
- Cuanto más grande el área, más saludable el perfil
- Ana (verde) tiene el área máxima - Flourishing
- Laura (rojo) tiene el área mínima - Crítico
- David (azul) destaca en "Sin Burnout" pero es bajo en "Sin Boreout"
- María (naranja) destaca en "Sin Boreout" pero es bajo en "Sin Burnout"



## 5. 🏛️ Perfil Cultural por Persona

Cada persona percibe la cultura de su empresa de forma diferente.
Con 8 preguntas (2 por cultura), el diagnóstico es más robusto.

In [6]:

# HEATMAP DE CULTURA PERCIBIDA


culturas_data = []
for c in resultados:
    kpis = c["_kpis"]
    if "cultura_scores" not in kpis:
        continue
    culturas_data.append({
        "Persona": c["Nombre"].split(" (")[0],
        "Perfil": c["Perfil"],
        "🔵 Adhocracia": kpis["cultura_scores"]["Adhocracia"],
        "🟢 Clan": kpis["cultura_scores"]["Clan"],
        "🟠 Mercado": kpis["cultura_scores"]["Mercado"],
        "🟡 Jerárquica": kpis["cultura_scores"]["Jerarquica"]
    })

df_culturas = pd.DataFrame(culturas_data)

fig = px.imshow(
    df_culturas[["🔵 Adhocracia", "🟢 Clan", "🟠 Mercado", "🟡 Jerárquica"]].values,
    x=["🔵 Adhocracia", "🟢 Clan", "🟠 Mercado", "🟡 Jerárquica"],
    y=[f"{r['Persona']} ({r['Perfil']})" for _, r in df_culturas.iterrows()],
    color_continuous_scale='Blues',
    zmin=1, zmax=5,
    title="🏛️ Cultura Percibida por Persona (8 preguntas)",
    text_auto='.1f',
    aspect='auto'
)

fig.update_layout(height=450, yaxis=dict(tickfont=dict(size=11)))
fig.show()

print("""
💡 INSIGHT CRUCIAL:
Cada perfil tiende a ciertas culturas:
- Ana (Flourishing): percibe cultura CLAN (colaboración)
- María (Quemada): percibe cultura MERCADO (resultados)
- David (Aburrido): percibe cultura JERÁRQUICA (normas)
- Laura (Crítica): percibe cultura MERCADO (presión)

Esto sugiere correlaciones reales entre cultura organizacional y bienestar.
""")


💡 INSIGHT CRUCIAL:
Cada perfil tiende a ciertas culturas:
- Ana (Flourishing): percibe cultura CLAN (colaboración)
- María (Quemada): percibe cultura MERCADO (resultados)
- David (Aburrido): percibe cultura JERÁRQUICA (normas)
- Laura (Crítica): percibe cultura MERCADO (presión)

Esto sugiere correlaciones reales entre cultura organizacional y bienestar.



## 6. 📈 Percentiles vs Benchmark

Cada persona se posiciona respecto al benchmark de 12,500 empleados.

In [7]:

# VISUALIZACIÓN DE PERCENTILES


fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "🔥 Burnout - Percentiles",
        "😴 Boreout - Percentiles",
        "💚 Bienestar - Percentiles",
       
    ),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "bar"}]]
)

kpis_a_mostrar = ["burnout", "boreout", "bienestar"]
colores_kpi = ["#e74c3c", "#3498db", "#2ecc71"]
posiciones = [(1,1), (1,2), (2,1)]

for i, (kpi, color, pos) in enumerate(zip(kpis_a_mostrar, colores_kpi, posiciones)):
    personas = [c["Nombre"].split(" (")[0] for c in resultados]
    percentiles = [c["_perfil"]["percentiles"][kpi] for c in resultados]
    
    fig.add_trace(
        go.Bar(
            x=personas,
            y=percentiles,
            marker_color=color,
            text=[f"P{p}" for p in percentiles],
            textposition='outside',
            name=kpi,
            showlegend=False
        ),
        row=pos[0], col=pos[1]
    )
    
    # Líneas de umbral
    fig.add_hline(y=25, line_dash="dot", line_color="green", opacity=0.5, row=pos[0], col=pos[1])
    fig.add_hline(y=50, line_dash="dot", line_color="yellow", opacity=0.5, row=pos[0], col=pos[1])
    fig.add_hline(y=75, line_dash="dot", line_color="red", opacity=0.5, row=pos[0], col=pos[1])

fig.update_layout(
    title_text="<b>📈 Posición de cada persona vs Benchmark (12,500 empleados)</b>",
    height=700,
    width=1100
)

for i in range(1, 3):
    for j in range(1, 3):
        fig.update_yaxes(range=[0, 100], row=i, col=j)

fig.show()

print("""
💡 INTERPRETACIÓN:
- Percentil 0-25: BAJO (mejor que el 75% del benchmark)
- Percentil 25-50: MEDIO-BAJO
- Percentil 50-75: MEDIO-ALTO
- Percentil 75-100: ALTO (peor que el 75% del benchmark)

⚠️ Para Burnout/Boreout: percentil ALTO = MALO
✅ Para Bienestar: percentil ALTO = BUENO
""")


💡 INTERPRETACIÓN:
- Percentil 0-25: BAJO (mejor que el 75% del benchmark)
- Percentil 25-50: MEDIO-BAJO
- Percentil 50-75: MEDIO-ALTO
- Percentil 75-100: ALTO (peor que el 75% del benchmark)

⚠️ Para Burnout/Boreout: percentil ALTO = MALO
✅ Para Bienestar: percentil ALTO = BUENO



## 6. 📥 Simulación: 30 Compañeros Responden

Generamos datos simulados de 30 compañeros con distribuciones realistas.

In [8]:

# GENERAR 30 COMPAÑEROS SIMULADOS


np.random.seed(42)
n_companeros = 30

perfiles_dist = {
    "flourishing": 0.15, "estable": 0.30, "quemado": 0.15,
    "aburrido": 0.20, "critico": 0.05, "vuelo": 0.15
}

# ✅ CORREGIDO: Todos los rangos tienen low < high
rangos = {
    "flourishing": {"burnout": (1, 3), "boreout": (1, 3), "bienestar": (4, 6), "rotacion": (1, 3)},
    "estable": {"burnout": (2, 4), "boreout": (2, 4), "bienestar": (3, 5), "rotacion": (2, 4)},
    "quemado": {"burnout": (4, 6), "boreout": (1, 3), "bienestar": (1, 3), "rotacion": (2, 4)},
    "aburrido": {"burnout": (1, 3), "boreout": (4, 6), "bienestar": (1, 3), "rotacion": (3, 5)},
    "critico": {"burnout": (4, 6), "boreout": (4, 6), "bienestar": (1, 2), "rotacion": (4, 6)},  # ← CORREGIDO
    "vuelo": {"burnout": (3, 5), "boreout": (3, 5), "bienestar": (2, 4), "rotacion": (4, 6)}
}

culturas_tipicas = {
    "flourishing": {"Adhocracia": (2, 5), "Clan": (4, 6), "Mercado": (1, 3), "Jerarquica": (1, 3)},
    "estable": {"Adhocracia": (2, 5), "Clan": (3, 5), "Mercado": (2, 5), "Jerarquica": (2, 5)},
    "quemado": {"Adhocracia": (1, 4), "Clan": (1, 4), "Mercado": (4, 6), "Jerarquica": (2, 5)},
    "aburrido": {"Adhocracia": (1, 3), "Clan": (1, 4), "Mercado": (2, 5), "Jerarquica": (4, 6)},
    "critico": {"Adhocracia": (1, 3), "Clan": (1, 3), "Mercado": (4, 6), "Jerarquica": (3, 6)},
    "vuelo": {"Adhocracia": (2, 5), "Clan": (2, 4), "Mercado": (3, 6), "Jerarquica": (2, 5)}
}

respuestas_sim = []
for i in range(n_companeros):
    perfil = np.random.choice(list(perfiles_dist.keys()), p=list(perfiles_dist.values()))
    r = rangos[perfil]
    c = culturas_tipicas[perfil]
    
    resp = {
        **{f'q{q}': np.random.randint(r["burnout"][0], r["burnout"][1]) for q in [16, 19, 23, 28]},
        **{f'q{q}': np.random.randint(r["boreout"][0], r["boreout"][1]) for q in [37, 39, 41, 43]},
        **{f'q{q}': np.random.randint(r["bienestar"][0], r["bienestar"][1]) for q in [45, 46, 47, 48]},
        **{f'q{q}': np.random.randint(r["rotacion"][0], r["rotacion"][1]) for q in [57, 58, 59]},
        'q65': np.random.randint(c["Adhocracia"][0], c["Adhocracia"][1]),
        'q66': np.random.randint(c["Adhocracia"][0], c["Adhocracia"][1]),
        'q67': np.random.randint(c["Clan"][0], c["Clan"][1]),
        'q68': np.random.randint(c["Clan"][0], c["Clan"][1]),
        'q69': np.random.randint(c["Mercado"][0], c["Mercado"][1]),
        'q70': np.random.randint(c["Mercado"][0], c["Mercado"][1]),
        'q71': np.random.randint(c["Jerarquica"][0], c["Jerarquica"][1]),
        'q72': np.random.randint(c["Jerarquica"][0], c["Jerarquica"][1])
    }
    respuestas_sim.append(resp)

datos_clasificados = []
for i, resp in enumerate(respuestas_sim):
    kpis = calcular_kpis_lite(resp)
    perfil = clasificar_individuo(kpis)
    datos_clasificados.append({
        "id": f"Compañero_{i+1:02d}",
        "perfil": perfil["perfil"],
        "perfil_nombre": perfil["nombre"],
        "burnout": round(kpis["burnout"], 2),
        "boreout": round(kpis["boreout"], 2),
        "bienestar": round(kpis["bienestar"], 2),
        "rotacion": round(kpis["rotacion"], 2),
        "cultura_dominante": kpis["cultura_dominante"]
    })

df_clasificados = pd.DataFrame(datos_clasificados)


print(f"👥 {n_companeros} COMPAÑEROS CLASIFICADOS")

print(df_clasificados[["id", "perfil_nombre", "cultura_dominante", 
                       "burnout", "boreout", "bienestar", "rotacion"]].head(10).to_string(index=False))
print(f"\n... y {len(df_clasificados)-10} más")

👥 30 COMPAÑEROS CLASIFICADOS
          id      perfil_nombre cultura_dominante  burnout  boreout  bienestar  rotacion
Compañero_01   🟡 Estable Neutro        Adhocracia     2.25     2.25       3.00      2.67
Compañero_02 🔵 Aburrido Crónico        Jerarquica     1.75     4.00       1.75      3.67
Compañero_03   🟠 Quemado Activo           Mercado     4.00     1.50       1.75      2.67
Compañero_04     🔴 Crítico Dual           Mercado     4.00     4.00       2.75      4.67
Compañero_05   🟡 Estable Neutro              Clan     2.50     2.50       3.50      3.00
Compañero_06   🟡 Estable Neutro        Adhocracia     2.75     2.00       3.25      2.33
Compañero_07      🟢 Flourishing              Clan     1.50     1.75       4.00      1.33
Compañero_08      🟢 Flourishing              Clan     1.50     1.25       4.50      1.67
Compañero_09   🟡 Estable Neutro              Clan     2.25     3.00       3.25      3.00
Compañero_10     ⚫ Desvinculado           Mercado     3.25     3.25       2.50   

## 7. 📊 Distribución de Perfiles

In [9]:

# DISTRIBUCIÓN DE PERFILES (PIE + BAR)


distribucion_perfiles = df_clasificados["perfil_nombre"].value_counts()
distribucion_culturas = df_clasificados["cultura_dominante"].value_counts()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"<b>Distribución de Perfiles ({n_companeros} personas)</b>",
        f"<b>Distribución de Culturas Percibidas</b>"
    ),
    specs=[[{"type": "pie"}, {"type": "pie"}]]
)

color_perfil = {
    "🟢 Flourishing": "#2ecc71", "🟡 Estable Neutro": "#f1c40f",
    "🟠 Quemado Activo": "#e67e22", "🔵 Aburrido Crónico": "#3498db",
    "🔴 Crítico Dual": "#e74c3c", "⚫ Vuelo Inminente": "#34495e"
}

color_cultura = {
    "Adhocracia": "#3498db", "Clan": "#2ecc71",
    "Mercado": "#e67e22", "Jerarquica": "#f1c40f"
}

fig.add_trace(go.Pie(
    labels=distribucion_perfiles.index,
    values=distribucion_perfiles.values,
    marker=dict(colors=[color_perfil.get(p, "#95a5a6") for p in distribucion_perfiles.index]),
    textinfo='label+percent', hole=0.3, textfont=dict(size=10)
), row=1, col=1)

fig.add_trace(go.Pie(
    labels=distribucion_culturas.index,
    values=distribucion_culturas.values,
    marker=dict(colors=[color_cultura.get(c, "#95a5a6") for c in distribucion_culturas.index]),
    textinfo='label+percent', hole=0.3, textfont=dict(size=11)
), row=1, col=2)

fig.update_layout(height=550, width=1100, showlegend=False, margin=dict(t=80, b=40))
fig.show()

print("\n📋 RESUMEN DE PERFILES:")
for perfil, count in distribucion_perfiles.items():
    pct = count / len(df_clasificados) * 100
    print(f"   {perfil:25s}: {count:2d} ({pct:4.1f}%)")

print("\n🏛️ RESUMEN DE CULTURAS:")
for cultura, count in distribucion_culturas.items():
    pct = count / len(df_clasificados) * 100
    print(f"   {cultura:12s}: {count:2d} ({pct:4.1f}%)")


📋 RESUMEN DE PERFILES:
   🟡 Estable Neutro         : 10 (33.3%)
   🔴 Crítico Dual           :  6 (20.0%)
   🔵 Aburrido Crónico       :  5 (16.7%)
   🟢 Flourishing            :  4 (13.3%)
   🟠 Quemado Activo         :  3 (10.0%)
   ⚫ Desvinculado           :  2 ( 6.7%)

🏛️ RESUMEN DE CULTURAS:
   Clan        : 11 (36.7%)
   Mercado     : 10 (33.3%)
   Jerarquica  :  5 (16.7%)
   Adhocracia  :  4 (13.3%)


## 8. 🗺️ Mapa de Posicionamiento Individual

Todos los compañeros en el espacio Burnout vs Boreout:

In [10]:

# SCATTER: POSICIONAMIENTO INDIVIDUAL

color_perfil = {
    "🟢 Flourishing": "#2ecc71", "🟡 Estable Neutro": "#f1c40f",
    "🟠 Quemado Activo": "#e67e22", "🔵 Aburrido Crónico": "#3498db",
    "🔴 Crítico Dual": "#e74c3c", "⚫ Vuelo Inminente": "#34495e"
}

fig = px.scatter(
    df_clasificados,
    x="burnout", y="boreout",
    color="perfil_nombre",
    color_discrete_map= color_perfil,
    symbol="cultura_dominante",
    size_max=15,
    hover_data=["id", "bienestar", "rotacion"],
    title="🗺️ Mapa de Posicionamiento: Burnout vs Boreout",
    labels={"burnout": "Burnout →", "boreout": "Boreout →"}
)

fig.add_hline(y=3.5, line_dash="dash", line_color="red", opacity=0.4,
              annotation_text="Umbral Boreout Alto (3.5)")
fig.add_vline(x=3.5, line_dash="dash", line_color="red", opacity=0.4,
              annotation_text="Umbral Burnout Alto (3.5)")

fig.add_annotation(x=1.5, y=1.5, text="🟢 Flourishing", showarrow=False, 
                   font_size=13, font=dict(color="#2ecc71", family="Arial Black"))
fig.add_annotation(x=4.5, y=1.5, text="🟠 Quemado", showarrow=False, 
                   font_size=13, font=dict(color="#e67e22", family="Arial Black"))
fig.add_annotation(x=1.5, y=4.5, text="🔵 Aburrido", showarrow=False, 
                   font_size=13, font=dict(color="#3498db", family="Arial Black"))
fig.add_annotation(x=4.5, y=4.5, text="🔴 Crítico", showarrow=False, 
                   font_size=13, font=dict(color="#e74c3c", family="Arial Black"))

fig.update_layout(
    height=650, width=1000,
    xaxis_range=[1, 5.2], yaxis_range=[1, 5.2],
    legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5)
)
fig.show()

print("""
💡 INTERPRETACIÓN:
- Cada PUNTO es una persona
- COLOR = perfil de bienestar
- FORMA = cultura percibida de su empresa
- Los 4 cuadrantes representan los 4 estados organizacionales
""")


💡 INTERPRETACIÓN:
- Cada PUNTO es una persona
- COLOR = perfil de bienestar
- FORMA = cultura percibida de su empresa
- Los 4 cuadrantes representan los 4 estados organizacionales



## 9. 🏆 Insight Clave: Cultura vs Perfil

¿Existe correlación entre la cultura percibida y el perfil de bienestar?

In [11]:

# MATRIZ CULTURA × PERFIL (CROSSTAB)


crosstab = pd.crosstab(
    df_clasificados["cultura_dominante"],
    df_clasificados["perfil_nombre"],
    normalize='index'
) * 100

fig = px.imshow(
    crosstab,
    color_continuous_scale='YlOrRd',
    zmin=0, zmax=100,
    title="🏆 % de Perfiles por Cultura Percibida (Heatmap Cruzado)",
    text_auto='.0f',
    aspect='auto'
)

fig.update_layout(
    height=500,
    xaxis_title="Perfil de Bienestar",
    yaxis_title="Cultura Dominante",
    xaxis=dict(tickangle=-30)
)
fig.show()

print("""
 CONCLUSIONES CRUZADAS:

1. CULTURA JERÁRQUICA → Mayor % de Aburridos
   - Procesos rígidos + falta de autonomía = boreout
   
2. CULTURA DE MERCADO → Mayor % de Quemados y Críticos
   - Presión por resultados + competitividad = burnout
   
3. CULTURA CLAN → Mayor % de Flourishing
   - Apoyo + desarrollo + familia = bienestar
   
4. CULTURA ADHOCRACIA → Mayor % de Flourishing y Estables
   - Innovación + creatividad = estimulación

""")


 CONCLUSIONES CRUZADAS:

1. CULTURA JERÁRQUICA → Mayor % de Aburridos
   - Procesos rígidos + falta de autonomía = boreout

2. CULTURA DE MERCADO → Mayor % de Quemados y Críticos
   - Presión por resultados + competitividad = burnout

3. CULTURA CLAN → Mayor % de Flourishing
   - Apoyo + desarrollo + familia = bienestar

4. CULTURA ADHOCRACIA → Mayor % de Flourishing y Estables
   - Innovación + creatividad = estimulación


